# LongFlow — Gate Night 8 (the CFG repair test)

Runtime: **L4 GPU**. ~45 min, ~$2–3. Pre-registered criteria:
`experiments/p1_flow_head/NOTES.md` (Gate Night 8 entry).

The 2026-08-15 audit found the flow head has been sampling the UNGUIDED
field since July — `flow_sample` silently discarded `neg_condition` and
`cfg_scale` while the teacher generates with CFG 1.3 every frame. This
night measures how much of the closed-loop deficit inference-time guidance
(`CFGFlowHeadPatch`, shipped + unit-tested) repairs.

**Requires the repo at commit with `CFGFlowHeadPatch`** — the cold-start
cell asserts it imports.

| cell | what |
|---|---|
| 4 | CFG-corrected closed loop: euler4 + heun8 × seeds 0/1; plain heun8 control; euler4 control copied from GN6 |
| 6 | teacher at cfg_scale=1.0 (guidance-off teacher) |
| 8 | FD backfill |
| 10 | Bundle |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "GN8 v1.0 (2026-08-15): CFG repair test"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag bundle"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample, heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, FlowHeadPatch
from src.flow_head.trainer import load_checkpoint
print("CFGFlowHeadPatch import OK — repo is at the audit-fix commit or later")

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
GATE6_DIR = "/content/drive/MyDrive/longflow_gate6"
OUT = "/content/gate_night8"
DRIVE_OUT = "/content/drive/MyDrive/longflow_gate8"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_wav(tag, wav, extra=None):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    row = {"tag": tag, "audio_s": round(len(wav)/24000, 1)}
    if extra: row.update(extra)
    report["runs"].append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night8_report.json", "w"), indent=2)
    print(row, flush=True)

def done(tag):
    if os.path.exists(f"{DRIVE_OUT}/{tag}.wav"):
        print(f"skip {tag}", flush=True)
        return True
    return False

report = {"runs": [], "notebook_version": NOTEBOOK_VERSION}
print(f"READY — {NOTEBOOK_VERSION}")


In [ ]:
# shared inputs — same construction as GN5–GN7 (identical script);
# euler4 plain control copied from GN6 (byte-identical settings, no GPU cost)
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern} — check Drive mount/paths")

sents = []
for f in drive_glob(f"{TRAIN_CACHE_DIR}/*.pt")[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
prompts = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")
P0 = prompts[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS = []
w = 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800: break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("script:", w, "words")
report["sweep_words"] = w
report["sweep_script"] = ABL_SCRIPT

if not done("c8_euler4_plain_s0"):
    src_wav = f"{GATE6_DIR}/g6_sig000.wav"
    assert os.path.exists(src_wav), "GN6 base render missing from Drive"
    shutil.copy(src_wav, f"{DRIVE_OUT}/c8_euler4_plain_s0.wav")
    shutil.copy(src_wav, f"{OUT}/c8_euler4_plain_s0.wav")
    report["runs"].append({"tag": "c8_euler4_plain_s0", "copied_from": "g6_sig000"})
    json.dump(report, open(f"{DRIVE_OUT}/gate_night8_report.json", "w"), indent=2)
    print("copied g6_sig000 -> c8_euler4_plain_s0")


## 4. CFG-corrected closed loop + plain heun8 control

`CFGFlowHeadPatch` honors the `neg_condition`/`cfg_scale=1.3` VibeVoice
passes on every frame; the plain-control arm uses the old `FlowHeadPatch`
(unguided). Everything else identical to GN4–GN7 closed-loop runs.


In [ ]:
HEAD_ARMS = [
    ("c8_euler4_s0", euler_sample, 4, 0, True),
    ("c8_euler4_s1", euler_sample, 4, 1, True),
    ("c8_heun8_s0", heun_sample, 8, 0, True),
    ("c8_heun8_s1", heun_sample, 8, 1, True),
    ("c8_heun8_plain_s0", heun_sample, 8, 0, False),
]

for tag, sampler, nfe, seed, use_cfg in HEAD_ARMS:
    if done(tag):
        continue
    torch.manual_seed(seed)
    cls = CFGFlowHeadPatch if use_cfg else FlowHeadPatch
    with cls(model, head20, mean20, std20, nfe=nfe, sway=0.0,
             sampler=sampler) as patch, torch.inference_mode():
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"cfg": use_cfg, "nfe": nfe, "seed": seed,
                        "latent_std": round(float(zs.std()), 3),
                        "frames": patch.calls})


## 6. Teacher at cfg_scale=1.0

How load-bearing is guidance for the TEACHER? Same script, stock DDPM head,
guidance off. Reference clean teacher at 1.3 = GN7 `t7_sig000`.


In [ ]:
tag = "t8_cfg10"
if not done(tag):
    torch.manual_seed(0)
    with torch.inference_mode():
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.0, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    save_wav(tag, wav, {"cfg_scale": 1.0})


## 8. FD backfill — all renders vs the GN3 teacher reference


In [ ]:
D_LAT = std20.numel()

def unwrap(z):
    while isinstance(z, (tuple, list)):
        z = z[0]
    if hasattr(z, "sample"):
        z = z.sample() if callable(z.sample) else z.sample
    while isinstance(z, (tuple, list)):
        z = z[0]
    assert torch.is_tensor(z), f"could not unwrap encoder output: {type(z)}"
    return z

def encode_latents(path):
    x, sr = sf.read(path, dtype="float32")
    z_all = []
    step = 24000 * 30
    with torch.inference_mode():
        for i in range(0, len(x), step):
            seg = torch.from_numpy(x[i:i+step])[None, None].to("cuda", torch.bfloat16)
            z = unwrap(model.model.acoustic_tokenizer.encode(seg))
            while z.ndim > 2:
                z = z.squeeze(0)
            if z.shape[-1] != D_LAT and z.shape[0] == D_LAT:
                z = z.T
            assert z.shape[-1] == D_LAT, f"latent width {z.shape} vs ckpt {D_LAT}"
            z_all.append(z.float().cpu())
    out = torch.cat(z_all)
    print(f"  {os.path.basename(path)}: {len(out)} frames ({len(out)/7.5:.0f}s)", flush=True)
    return out

def fd_curve(z, ref_mu, ref_cov, win=75):
    import scipy.linalg
    out = []
    for i in range(0, len(z) - win, win):
        w = z[i:i+win].numpy()
        mu, cov = w.mean(0), np.cov(w.T)
        d = mu - ref_mu
        covmean = scipy.linalg.sqrtm(cov @ ref_cov)
        if np.iscomplexobj(covmean): covmean = covmean.real
        out.append(float(d @ d + np.trace(cov + ref_cov - 2*covmean)))
    return out

teacher_z = encode_latents(f"{GATE3_DIR}/t1_turnsplit_p0.wav")
half = len(teacher_z) // 2
ref_mu, ref_cov = teacher_z[:half].numpy().mean(0), np.cov(teacher_z[:half].numpy().T)

TAGS = ["c8_euler4_s0", "c8_euler4_s1", "c8_heun8_s0", "c8_heun8_s1",
        "c8_euler4_plain_s0", "c8_heun8_plain_s0", "t8_cfg10"]
fd = {"teacher_self": fd_curve(teacher_z[half:], ref_mu, ref_cov)}
for tag in TAGS:
    p = f"{DRIVE_OUT}/{tag}.wav"
    if os.path.exists(p):
        fd[tag] = fd_curve(encode_latents(p), ref_mu, ref_cov)
report["fd_curves"] = fd
json.dump(report, open(f"{DRIVE_OUT}/gate_night8_report.json", "w"), indent=2)
for k, v in fd.items():
    print(f"{k}: first={v[0]:.1f} med={sorted(v)[len(v)//2]:.1f} last={v[-1]:.1f} n={len(v)}")


## 10. Bundle

`gate_night8_bundle.zip` (zips from the Drive mirror). Mac:
`unzip -o ~/Downloads/gate_night8_bundle.zip -d experiments/p1_flow_head/audio/gate_night8`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night8.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{DRIVE_OUT}/gate_night8_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night8_bundle.zip", "w") as z:
    for f in glob.glob(f"{DRIVE_OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night8_bundle.zip")
